# Evaluate Qwen3.5-0.8B LoRA Cloud Classifier — Classification

This notebook loads a LoRA adapter, runs inference on the test set using
single-label classification with `<answer>` tags, computes top-1 accuracy and
format rate, and saves all evaluation artifacts.


## 1. Install Dependencies (run once, then restart kernel)

In [ ]:
"""
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "--no-cache-dir", *pkgs])

# Remove conflicting packages first
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y",
                 "verl", "vllm", "mistral-common", "trl",
                 "peft", "transformers", "tokenizers", "accelerate", "bitsandbytes"])

pip_install("pip<25.1")

# Install transformers from GitHub dev (supports Qwen3.5 model_type="qwen3_5")
pip_install("git+https://github.com/huggingface/transformers.git")

# Install peft from GitHub dev (compatible with latest transformers)
pip_install("git+https://github.com/huggingface/peft.git",
            "accelerate>=1.0.0", "datasets>=3.0.0", "safetensors>=0.4.0",
            "scikit-learn>=1.6.0", "seaborn>=0.13.0", "matplotlib>=3.8.0,<3.11.0",
            "pillow>=10.4.0,<12.0.0", "torchvision", "qwen-vl-utils>=0.0.10")

subprocess.check_call([sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-cache-dir",
                       "numpy==2.1.3", "scipy==1.14.1", "scikit-learn==1.7.2", "pillow>=10.4.0,<12.0.0"])

# torchaudio's compiled extension mismatches the torch/CUDA build on Kaggle, which
# crashes transformers on import (it loads torchaudio via loss utils). We don't need
# it for vision, and transformers skips it automatically when it's absent.
subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "torchao", "torchaudio"])

print("Done. Restart kernel now — do NOT test imports here (run cell 2 after restart).")
"""

## 2. Configuration & Environment

In [ ]:
import ctypes, glob, os, re, sys, types, math, random, gc, json, warnings
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore", category=FutureWarning)

os.environ["PYTORCH_JIT"] = "0"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ.setdefault("WANDB_DISABLED", "true")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import site as _site
_patterns = ["/usr/local/cuda*/lib64/libnvrtc-builtins.so*", "/usr/local/cuda*/lib64/libnvJitLink.so*",
             "/usr/lib/x86_64-linux-gnu/libnvrtc-builtins.so*", "/usr/lib/x86_64-linux-gnu/libnvJitLink.so*"]
for _sp in _site.getsitepackages():
    _patterns += [os.path.join(_sp, "nvidia", "*", "lib", "libnvrtc-builtins.so*"),
                  os.path.join(_sp, "nvidia", "*", "lib", "libnvJitLink.so*")]
for _p in _patterns:
    for _lib in sorted(glob.glob(_p)):
        try: ctypes.CDLL(_lib, mode=ctypes.RTLD_GLOBAL)
        except OSError: pass

import torch, numpy as np
if hasattr(torch._C, "_jit_set_nvfuser_enabled"): torch._C._jit_set_nvfuser_enabled(False)
try:
    import triton.backends as _tb
    if not hasattr(_tb, "compiler"): _tb.compiler = types.SimpleNamespace(AttrsDescriptor=None)
except ImportError: pass

import transformers
# Newer transformers dev no longer exposes BloomPreTrainedModel, which peft imports
# at module load. Provide a dummy so `from transformers import BloomPreTrainedModel`
# inside peft succeeds (it's unused for LoRA adapter loading).
try:
    transformers.BloomPreTrainedModel
except Exception:
    transformers.BloomPreTrainedModel = type("BloomPreTrainedModel", (), {})
import peft
print(f"torch: {torch.__version__}\ntransformers: {transformers.__version__}\npeft: {peft.__version__}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU: {torch.cuda.get_device_name(0)} ({gpu.total_memory / 1024**3:.1f} GB)")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# ── CONFIGURE THESE PATHS ──
MODEL_NAME = "Qwen/Qwen3.5-0.8B"
DATASET_PATH = "/kaggle/input/datasets/nachomorera/clouddataset-fixed/CloudDataset_Fixed"
ADAPTER_PATH = "/kaggle/input/your-lora-adapter-dataset"  # <-- SET THIS to your adapter dataset path
OUTPUT_DIR = "/kaggle/working/eval_results"
EVAL_SPLIT = "test"
MAX_SAMPLES = None   # Set to e.g. 50 for quick test
SKIP_TOKEN_ACCURACY = False  # Set True to skip SFT-style token accuracy
TAG = "eval"
SEED = 42
MAX_LENGTH = 768

# Auto-detect adapter
_inp = Path("/kaggle/input")
if _inp.exists():
    print("\n=== /kaggle/input datasets ===")
    for d in sorted(_inp.iterdir()): print(f"  {d.name}/")
    candidates = list(_inp.rglob("adapter_model.safetensors"))
    if candidates:
        ADAPTER_PATH = str(candidates[0].parent)
        print(f"\nAuto-detected adapter: {ADAPTER_PATH}")

torch.manual_seed(SEED); np.random.seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\nModel:   {MODEL_NAME}")
print(f"Adapter: {ADAPTER_PATH}")
print(f"Dataset: {DATASET_PATH}")
print(f"Split:   {EVAL_SPLIT}")
print(f"Output:  {OUTPUT_DIR}")

assert os.path.isdir(DATASET_PATH), f"Dataset not found: {DATASET_PATH}"
assert os.path.isdir(ADAPTER_PATH), f"Adapter not found: {ADAPTER_PATH}"

## 3. Load & Merge Dataset

In [ ]:
from datasets import ClassLabel, load_dataset

print("Loading dataset...")
dataset = load_dataset("imagefolder", data_files={
    "train": f"{DATASET_PATH}/train/**",
    "validation": f"{DATASET_PATH}/val/**",
    "test": f"{DATASET_PATH}/test/**",
})

original_labels = dataset["train"].features["label"].names
print(f"Original classes ({len(original_labels)}): {original_labels}")

# WMO-aligned taxonomy: each descriptive label merged into its closest WMO genus.
class_mapping = {
    "Patterned Clouds":   "Altocumulus",
    "Sky (General)":      "Clear Sky",
    "Thick Dark Clouds":  "Cumulonimbus",
    "Thick White Clouds": "Cumulus",
    "Veil Clouds":        "Veil",
}
merged_labels = sorted({class_mapping.get(l, l) for l in original_labels})
label2id = {l: i for i, l in enumerate(merged_labels)}
NUM_LABELS = len(merged_labels)

def remap_example(ex):
    ex["label"] = label2id[class_mapping.get(original_labels[ex["label"]], original_labels[ex["label"]])]
    return ex

dataset = dataset.map(remap_example)
new_features = dataset["train"].features.copy()
new_features["label"] = ClassLabel(names=merged_labels)
dataset = dataset.cast(new_features)

new_labels = dataset["train"].features["label"].names
label2id = {l: i for i, l in enumerate(new_labels)}
id2label = {i: l for l, i in label2id.items()}
VALID_CLASSES_LOWER = {l.lower(): l for l in new_labels}

print(f"Merged classes ({NUM_LABELS}): {new_labels}")
print(f"Train: {len(dataset['train'])}, Val: {len(dataset['validation'])}, Test: {len(dataset['test'])}")


## 4. Classification Prompt & Parsing Helpers


In [ ]:
SYSTEM_PROMPT = (
    "You are a cloud classification expert with a lot of expertise on weather, physics, and clouds. "
    "Your goal is to, given a cloud image, classify it with one of the following classes:\n\n"
    + "\n".join(f"- {label}" for label in new_labels)
)

USER_PROMPT = (
    "Classify the cloud type in this image. "
    "The class must be included between the tags <answer></answer> using exactly one "
    "of the class names listed above. The tags must be placed at the end of your answer."
)

ANSWER_TEMPLATE = "<answer>{label}</answer>"


def build_prompt_messages():
    return [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user",   "content": [{"type": "image"}, {"type": "text", "text": USER_PROMPT}]},
    ]


def build_full_messages(label_text):
    """Prompt + assistant answer (used for teacher-forced token accuracy)."""
    answer = ANSWER_TEMPLATE.format(label=label_text)
    return build_prompt_messages() + [
        {"role": "assistant", "content": [{"type": "text", "text": answer}]},
    ]


def normalize_prediction(text):
    """Extract predicted class from <answer> tags with fallbacks."""
    m = re.search(r"<answer>\s*(.*?)\s*</answer>", text, re.IGNORECASE | re.DOTALL)
    if m:
        text = m.group(1).strip()
    cleaned = " ".join(text.replace("\n", " ").split()).strip().lower()
    if cleaned in VALID_CLASSES_LOWER:
        return VALID_CLASSES_LOWER[cleaned]
    for label in new_labels:
        if label.lower() in cleaned or cleaned in label.lower():
            return label
    cleaned_tokens = set(cleaned.replace("/", " ").replace("_", " ").replace("-", " ").split())
    best_label, best_score = None, 0
    for label in new_labels:
        score = len(cleaned_tokens & set(label.lower().replace("_", " ").replace("-", " ").split()))
        if score > best_score:
            best_score, best_label = score, label
    return best_label if best_label is not None else new_labels[0]


def has_valid_answer(text):
    """True if output contains a valid <answer> tag."""
    m = re.search(r"<answer>\s*(.*?)\s*</answer>", text, re.IGNORECASE | re.DOTALL)
    if not m:
        return False
    return " ".join(m.group(1).split()).strip().lower() in VALID_CLASSES_LOWER


print("Prompts & parsing ready.")


## 5. Load Model + LoRA Adapter

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import PeftModel

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_NAME)

compute_dtype = torch.bfloat16 if (DEVICE == "cuda" and torch.cuda.is_bf16_supported()) else torch.float32

print(f"Loading base model ({compute_dtype})...")
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME, torch_dtype=compute_dtype,
    device_map="auto" if DEVICE == "cuda" else None,
)
if DEVICE != "cuda": base_model = base_model.to(DEVICE)

print(f"Loading LoRA adapter from {ADAPTER_PATH}...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH, is_trainable=False)
model.eval()
model.print_trainable_parameters()
print("Model ready for evaluation.")

## 6. Run Inference on Test Set

In [ ]:
from tqdm.auto import tqdm

def classify_image(image):
    image = image.convert("RGB")
    prompt_text = processor.apply_chat_template(build_prompt_messages(), tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt_text], images=[image], padding=True, truncation=True,
                       max_length=MAX_LENGTH, return_tensors="pt")
    inputs = {k: v.to(next(model.parameters()).device) if hasattr(v, "to") else v for k, v in inputs.items()}
    with torch.inference_mode():
        gen_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    gen_ids = gen_ids[:, inputs["input_ids"].shape[1]:]
    return processor.batch_decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

split = dataset[EVAL_SPLIT]
if MAX_SAMPLES: split = split.select(range(min(MAX_SAMPLES, len(split))))

true_labels_list, pred_labels_list, raw_responses = [], [], []

for example in tqdm(split, total=len(split), desc=f"Evaluating ({EVAL_SPLIT})"):
    raw = classify_image(example["image"])
    true_labels_list.append(id2label[int(example["label"])])
    pred_labels_list.append(normalize_prediction(raw))
    raw_responses.append(raw)

accuracy_top1 = sum(t == p for t, p in zip(true_labels_list, pred_labels_list)) / len(true_labels_list)
format_rate   = sum(has_valid_answer(r) for r in raw_responses) / len(raw_responses)

print(f"\n{'='*50}")
print(f"  Top-1 accuracy: {accuracy_top1:.4f}")
print(f"  Format rate:    {format_rate:.4f}")
print(f"  Samples:        {len(true_labels_list)}")
print(f"{'='*50}")


## 7. Teacher-Forced Token Accuracy (SFT-style metric)

In [ ]:
if not SKIP_TOKEN_ACCURACY:
    per_class_correct = {l: 0 for l in new_labels}
    per_class_total = {l: 0 for l in new_labels}
    total_correct = total_tokens = 0

    for example in tqdm(split, total=len(split), desc="Token accuracy (teacher-forced)"):
        image = example["image"].convert("RGB")
        true_label = id2label[int(example["label"])]

        prompt_text = processor.apply_chat_template(build_prompt_messages(), tokenize=False, add_generation_prompt=True)
        full_text   = processor.apply_chat_template(build_full_messages(true_label),  tokenize=False, add_generation_prompt=False)

        prompt_enc = processor(text=[prompt_text], images=[image], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
        full_enc   = processor(text=[full_text],   images=[image], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
        full_enc   = {k: v.to(next(model.parameters()).device) if hasattr(v, "to") else v for k, v in full_enc.items()}
        prompt_len = int(prompt_enc["attention_mask"][0].sum().item())

        with torch.inference_mode():
            outputs = model(**{k: v for k, v in full_enc.items() if k != "labels"})

        shift_logits = outputs.logits[:, :-1, :]
        shift_labels = full_enc["input_ids"][:, 1:]
        preds = shift_logits.argmax(dim=-1)

        mask = torch.zeros_like(shift_labels, dtype=torch.bool)
        if prompt_len - 1 < mask.shape[1]: mask[:, prompt_len-1:] = True
        if full_enc["attention_mask"] is not None:
            mask = mask & (full_enc["attention_mask"][:, 1:] == 1)

        correct = ((preds == shift_labels) & mask).sum().item()
        count   = mask.sum().item()
        total_correct += correct; total_tokens += count
        per_class_correct[true_label] += correct; per_class_total[true_label] += count

    token_acc_overall = total_correct / total_tokens if total_tokens > 0 else 0.0
    token_acc_per_class = {l: per_class_correct[l]/per_class_total[l]
                           if per_class_total[l] > 0 else float("nan") for l in new_labels}

    print(f"\nOverall token accuracy (teacher-forced): {token_acc_overall:.4f}")
    print(f"\n  {'Class':30s} {'Top-1 (gen)':>12s} {'Token (SFT)':>12s}")
    print(f"  {'─'*30} {'─'*12} {'─'*12}")
    for label in new_labels:
        mask_l = [t == label for t in true_labels_list]
        gen_acc = sum(1 for m, p, t in zip(mask_l, pred_labels_list, true_labels_list) if m and p == t) / max(sum(mask_l), 1)
        tok = token_acc_per_class[label]
        print(f"  {label:30s} {gen_acc:>12.4f} {tok:>12.4f}" if not np.isnan(tok) else f"  {label:30s} {gen_acc:>12.4f} {'N/A':>12s}")
else:
    token_acc_overall  = None
    token_acc_per_class = None
    print("Skipped token accuracy (SKIP_TOKEN_ACCURACY=True)")


## 8. Classification Report & Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix

report  = classification_report(true_labels_list, pred_labels_list, labels=new_labels, digits=3, zero_division=0)
bal_acc = balanced_accuracy_score(true_labels_list, pred_labels_list)
print(f"\nClassification Report\n{report}")
print(f"Balanced accuracy: {bal_acc:.4f}")

cm = confusion_matrix(true_labels_list, pred_labels_list, labels=new_labels)
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=new_labels, yticklabels=new_labels, linewidths=0.5, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion Matrix | Top-1={accuracy_top1:.3f} | Bal Acc={bal_acc:.3f} | Format={format_rate:.3f}")
plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0); plt.tight_layout()
cm_path = os.path.join(OUTPUT_DIR, f"confusion_matrix_{TAG}.png")
plt.savefig(cm_path, dpi=150); plt.show()
print(f"Saved: {cm_path}")


## 9. Per-Class Accuracy Comparison Chart

In [ ]:
per_class_gen = {}
for label in new_labels:
    mask = [t == label for t in true_labels_list]
    total = sum(mask)
    per_class_gen[label] = sum(1 for m, p, t in zip(mask, pred_labels_list, true_labels_list) if m and p == t) / total if total > 0 else 0.0

labels_list = list(per_class_gen.keys())
gen_vals    = [per_class_gen[l] for l in labels_list]

if token_acc_per_class is not None:
    tok_vals = [token_acc_per_class[l] if not np.isnan(token_acc_per_class.get(l, float("nan"))) else 0.0 for l in labels_list]
    y = np.arange(len(labels_list)); bh = 0.35
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(y - bh/2, gen_vals, bh, label="Top-1 (generation)",      color="#2196F3", edgecolor="black", linewidth=0.5)
    ax.barh(y + bh/2, tok_vals, bh, label="Token Acc (teacher-forced)", color="#FF9800", edgecolor="black", linewidth=0.5)
    ax.set_yticks(y); ax.set_yticklabels(labels_list)
    ax.set_xlabel("Accuracy"); ax.set_title("Per-Class: Generation Top-1 vs Token Accuracy")
    ax.set_xlim(0, 1.12); ax.legend(loc="lower right"); ax.invert_yaxis()
else:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(labels_list, gen_vals, color="#2196F3", edgecolor="black", linewidth=0.5)
    ax.set_xlabel("Top-1 Accuracy"); ax.set_title("Per-Class Top-1 Accuracy")
    ax.axvline(1 / NUM_LABELS, color="red", ls="--", label="Random baseline"); ax.legend()
    ax.set_xlim(0, 1.05); ax.invert_yaxis()
plt.tight_layout()
chart_path = os.path.join(OUTPUT_DIR, f"per_class_accuracy_{TAG}.png")
plt.savefig(chart_path, dpi=150); plt.show()
print(f"Saved: {chart_path}")


## 10. Predictions CSV


In [ ]:
# ── Infer source dataset from image path (best-effort) ───────────────────────
def _infer_source(raw_path: str) -> str:
    if not raw_path:
        return "unknown"
    p = raw_path.lower()
    if "ccsn" in p:      return "CCSN"
    if "fabra" in p and "swimcat" not in p: return "FabraClouds"
    if "fabraswimcat" in p or ("fabra" in p and "swimcat" in p): return "FabraSwimcat"
    if "swimcat" in p:   return "SWIMCAT"
    return "unknown"

test_split_paths = dataset[EVAL_SPLIT]
if MAX_SAMPLES:
    test_split_paths = test_split_paths.select(range(min(MAX_SAMPLES, len(test_split_paths))))

source_list = ["unknown"] * len(true_labels_list)
try:
    _raw = test_split_paths.data.column("image").to_pylist()
    source_list = [_infer_source(r.get("path", "") if isinstance(r, dict) else "") for r in _raw]
except Exception:
    pass

predictions_df = pd.DataFrame({
    "source_dataset": source_list,
    "true_label":     true_labels_list,
    "predicted":      pred_labels_list,
    "correct":        [t == p for t, p in zip(true_labels_list, pred_labels_list)],
    "raw_response":   raw_responses,
})
pred_path = os.path.join(OUTPUT_DIR, f"predictions_{TAG}.csv")
predictions_df.to_csv(pred_path, index=False)
print(f"Saved predictions: {pred_path}")
print(f"Total samples: {len(predictions_df)}, correct: {predictions_df['correct'].sum()}")


## 11. Save Metrics & Zip Artifacts

In [ ]:
import shutil

metrics_txt = os.path.join(OUTPUT_DIR, f"metrics_{TAG}.txt")
with open(metrics_txt, "w") as f:
    f.write(f"Top-1 accuracy:    {accuracy_top1:.6f}\n")
    f.write(f"Balanced accuracy: {bal_acc:.6f}\n")
    f.write(f"Format rate:       {format_rate:.6f}\n")
    if token_acc_overall is not None:
        f.write(f"Token accuracy (teacher-forced): {token_acc_overall:.6f}\n")
    f.write(f"Samples: {len(true_labels_list)}\n")
    f.write(f"Model:   {MODEL_NAME}\n")
    f.write(f"Adapter: {ADAPTER_PATH}\n\n")
    f.write(report)
print(f"Metrics saved: {metrics_txt}")

_kappa = kappa if "kappa" in dir() else None
metrics_json = os.path.join(OUTPUT_DIR, f"metrics_{TAG}.json")
with open(metrics_json, "w") as f:
    json.dump({
        "top1":             accuracy_top1,
        "balanced_accuracy": bal_acc,
        "cohen_kappa":      _kappa,
        "format_rate":      format_rate,
        "token_accuracy":   token_acc_overall,
        "n_samples":        len(true_labels_list),
        "model":            MODEL_NAME,
        "adapter":          ADAPTER_PATH,
        "timestamp":        datetime.now().isoformat(),
    }, f, indent=2, default=str)
print(f"Metrics JSON saved: {metrics_json}")

zip_path = os.path.join("/kaggle/working", f"eval_artifacts_{TAG}")
shutil.make_archive(zip_path, "zip", OUTPUT_DIR)
print(f"\nAll artifacts zipped: {zip_path}.zip")
print(f"\n=== FINAL RESULTS ===")
print(f"Top-1: {accuracy_top1:.4f} | Bal Acc: {bal_acc:.4f} | Format: {format_rate:.4f}")
if token_acc_overall is not None:
    print(f"Token Acc (SFT): {token_acc_overall:.4f}")
if _kappa is not None:
    print(f"Cohen kappa: {_kappa:.4f}")


## 12. Additional Classification Metrics (Cohen's κ, Per-Class F1)


In [ ]:
from sklearn.metrics import cohen_kappa_score, f1_score

kappa = cohen_kappa_score(true_labels_list, pred_labels_list)
f1_per_class = f1_score(true_labels_list, pred_labels_list, labels=new_labels, average=None, zero_division=0)
f1_macro  = f1_score(true_labels_list, pred_labels_list, average="macro",  zero_division=0)
f1_weighted = f1_score(true_labels_list, pred_labels_list, average="weighted", zero_division=0)

print(f"Cohen's kappa   : {kappa:.4f}  (0=random, 1=perfect)")
print(f"F1 macro        : {f1_macro:.4f}")
print(f"F1 weighted     : {f1_weighted:.4f}")
print()
print(f"  {'Class':30s} {'Top-1':>7s} {'F1':>7s}")
print("─" * 50)
for label, f1 in zip(new_labels, f1_per_class):
    t1 = per_class_gen.get(label, 0.0)
    print(f"  {label:28s} {t1:>7.3f} {f1:>7.3f}")
print("─" * 50)
print(f"  {'MACRO AVG':28s} {accuracy_top1:>7.3f} {f1_macro:>7.3f}")


## 13. Hard Confusion Pairs (Visually Similar Classes)

In [ ]:
# Visually ambiguous WMO class pairs
HARD_PAIRS = [
    ("Altocumulus",   "Cirrocumulus"),
    ("Cirrocumulus",  "Altocumulus"),
    ("Altocumulus",   "Stratocumulus"),
    ("Stratocumulus", "Altocumulus"),
    ("Stratocumulus", "Stratus"),
    ("Stratus",       "Stratocumulus"),
    ("Altostratus",   "Stratus"),
    ("Stratus",       "Altostratus"),
    ("Cumulonimbus",  "Cumulus"),
    ("Cumulus",       "Cumulonimbus"),
    ("Cirrus",        "Veil"),
    ("Veil",          "Cirrus"),
    ("Cirrus",        "Altostratus"),
    ("Altostratus",   "Cirrus"),
]

cm_df = pd.DataFrame(cm, index=new_labels, columns=new_labels)

print("Hard confusion pairs (true -> predicted errors)")
print(f"  {'True class':25s} -> {'Predicted class':25s} {'Wrong N':>8s} {'of Total':>9s} {'Err %':>8s}")
print("  " + "─" * 80)
for true_cls, pred_cls in HARD_PAIRS:
    if true_cls not in cm_df.index or pred_cls not in cm_df.columns:
        continue
    n_wrong = int(cm_df.loc[true_cls, pred_cls])
    n_total = int(cm_df.loc[true_cls].sum())
    err_pct = 100 * n_wrong / n_total if n_total else 0.0
    if n_wrong > 0:
        print(f"  {true_cls:25s} -> {pred_cls:25s} {n_wrong:>8d} {n_total:>9d} {err_pct:>7.1f}%")

hard_classes = sorted({c for pair in HARD_PAIRS for c in pair if c in new_labels})
sub_cm = cm_df.loc[hard_classes, hard_classes]
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(sub_cm, annot=True, fmt="d", cmap="Reds", linewidths=0.5, ax=ax,
            cbar_kws={"label": "Count"})
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("Confusion sub-matrix — visually similar class pairs")
plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0); plt.tight_layout()
hc_path = os.path.join(OUTPUT_DIR, f"hard_pairs_confusion_{TAG}.png")
plt.savefig(hc_path, dpi=150); plt.show()
print(f"Saved: {hc_path}")


## 14. Per-Class Precision / Recall / F1 Heatmap


In [ ]:
from sklearn.metrics import precision_recall_fscore_support

precisions, recalls, f1s, supports = precision_recall_fscore_support(
    true_labels_list, pred_labels_list, labels=new_labels, zero_division=0
)

prf_df = pd.DataFrame({
    "Precision": precisions,
    "Recall":    recalls,
    "F1":        f1s,
    "Support":   supports,
}, index=new_labels)

print(prf_df.to_string(float_format="{:.3f}".format))

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(prf_df[["Precision", "Recall", "F1"]], annot=True, fmt=".3f",
            cmap="YlGn", vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title(f"Per-Class Precision / Recall / F1 — {TAG}")
ax.set_xlabel("Metric"); ax.set_ylabel("Class")
plt.tight_layout()
prf_path = os.path.join(OUTPUT_DIR, f"per_class_prf_{TAG}.png")
plt.savefig(prf_path, dpi=150); plt.show()
print(f"Saved: {prf_path}")


## 15. Qualitative Failure Analysis — Misclassified Image Gallery

In [ ]:
import textwrap, collections
import random as _rng

N_EXAMPLES_PER_CLASS = 2
GALLERY_SEED = 0
_rng.seed(GALLERY_SEED)

wrong_by_class = collections.defaultdict(list)
for i, (t, p) in enumerate(zip(true_labels_list, pred_labels_list)):
    if t != p:
        wrong_by_class[t].append(i)

sampled_wrong = []
for label in new_labels:
    idxs = wrong_by_class[label]
    sampled_wrong.extend(_rng.sample(idxs, min(N_EXAMPLES_PER_CLASS, len(idxs))))
sampled_wrong.sort()

if not sampled_wrong:
    print("No misclassifications found — perfect accuracy!")
else:
    ncols = 4
    nrows = math.ceil(len(sampled_wrong) / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 4))
    axes = axes.flatten() if nrows > 1 or ncols > 1 else [axes]

    test_split = dataset[EVAL_SPLIT]
    if MAX_SAMPLES:
        test_split = test_split.select(range(min(MAX_SAMPLES, len(test_split))))

    for plot_idx, sample_idx in enumerate(sampled_wrong):
        ax   = axes[plot_idx]
        img  = test_split[sample_idx]["image"].convert("RGB")
        true = true_labels_list[sample_idx]
        pred = pred_labels_list[sample_idx]
        raw  = raw_responses[sample_idx]

        ax.imshow(img); ax.axis("off")
        title = f"TRUE: {true}\nPRED: {pred}"
        ax.set_title(textwrap.fill(title, 28), fontsize=8, color="darkred", fontweight="bold")

    for ax in axes[len(sampled_wrong):]:
        ax.set_visible(False)

    fig.suptitle(f"Misclassified Samples (up to {N_EXAMPLES_PER_CLASS} per class)", fontsize=13)
    plt.tight_layout()
    gallery_path = os.path.join(OUTPUT_DIR, f"failure_gallery_{TAG}.png")
    plt.savefig(gallery_path, dpi=120, bbox_inches="tight"); plt.show()
    print(f"Saved: {gallery_path}  |  Total shown: {len(sampled_wrong)}")
    print(f"Total misclassifications: {sum(len(v) for v in wrong_by_class.values())} / {len(true_labels_list)}")


## 16. Overfitting Diagnosis (Train vs Test Gap)

In [ ]:
OVERFITTING_CHECK = True
N_TRAIN_PER_CLASS = 10   # 10 x 11 = 110 forward passes

if not OVERFITTING_CHECK:
    print("Overfitting check skipped (OVERFITTING_CHECK=False).")
else:
    import random as _r, collections as _col
    _r.seed(SEED)

    train_split = dataset["train"]
    class_to_train_idx = _col.defaultdict(list)
    for i, ex in enumerate(train_split):
        class_to_train_idx[id2label[int(ex["label"])]].append(i)

    sampled_train_idx = []
    for label in new_labels:
        pool = class_to_train_idx[label]
        sampled_train_idx.extend(_r.sample(pool, min(N_TRAIN_PER_CLASS, len(pool))))

    train_true, train_pred = [], []
    for idx in tqdm(sampled_train_idx, desc="Overfitting check (train sample)"):
        ex  = train_split[idx]
        raw = classify_image(ex["image"])
        train_true.append(id2label[int(ex["label"])])
        train_pred.append(normalize_prediction(raw))

    train_top1 = sum(t == p for t, p in zip(train_true, train_pred)) / len(train_true)
    test_top1  = accuracy_top1

    train_pc = {}
    for label in new_labels:
        idxs = [i for i, t in enumerate(train_true) if t == label]
        train_pc[label] = sum(1 for i in idxs if train_pred[i] == label) / len(idxs) if idxs else float("nan")

    print("\n==== Overfitting Diagnostic ====")
    print(f"{'Metric':22s} {'Train (sample)':>16s} {'Test (full)':>14s} {'Gap (train-test)':>18s}")
    print("─" * 72)
    print(f"{'Top-1 accuracy':22s} {train_top1:>16.4f} {test_top1:>14.4f} {train_top1-test_top1:>+18.4f}")
    print()
    print("Per-class top-1 — Train sample vs Test full:")
    print(f"  {'Class':30s} {'Train':>7s} {'Test':>7s} {'Gap':>8s}")
    print("  " + "─" * 55)
    for label in new_labels:
        tr  = train_pc.get(label, float("nan"))
        te  = per_class_gen.get(label, float("nan"))
        gap = tr - te if not (np.isnan(tr) or np.isnan(te)) else float("nan")
        flag = "  <- overfit?" if (not np.isnan(gap) and gap > 0.20) else ""
        print(f"  {label:30s} {tr:>7.3f} {te:>7.3f} {gap:>+8.3f}{flag}")

    print("\nInterpretation: gap >0.10 = mild | gap >0.20 = significant ⚠")

    fig, ax = plt.subplots(figsize=(12, 6))
    y  = np.arange(NUM_LABELS); bh = 0.35
    te_vals = [per_class_gen.get(l, 0) for l in new_labels]
    tr_vals = [train_pc.get(l, 0)      for l in new_labels]
    ax.barh(y - bh/2, tr_vals, bh, label=f"Train sample (n={N_TRAIN_PER_CLASS}/class)", color="#E74C3C", alpha=0.85, edgecolor="black", lw=0.5)
    ax.barh(y + bh/2, te_vals, bh, label="Test full (n=all)", color="#2980B9", alpha=0.85, edgecolor="black", lw=0.5)
    ax.set_yticks(y); ax.set_yticklabels(new_labels)
    ax.set_xlabel("Top-1 Accuracy")
    ax.set_title(f"Overfitting Check: Train vs Test Top-1 per Class\nOverall gap = {train_top1-test_top1:+.3f} (train={train_top1:.3f}, test={test_top1:.3f})")
    ax.set_xlim(0, 1.15); ax.legend(); ax.invert_yaxis(); plt.tight_layout()
    overfit_path = os.path.join(OUTPUT_DIR, f"overfitting_check_{TAG}.png")
    plt.savefig(overfit_path, dpi=150); plt.show()
    print(f"Saved: {overfit_path}")


## 17. Zero-Shot Baseline (Qwen3.5-0.8B, No Adapter, Same Classification Prompt)

Compare the LoRA-adapted model against the **unmodified base model** using the
*exact same classification prompt*, so the comparison is apples-to-apples.


In [ ]:
ZERO_SHOT_CHECK = True
N_ZS_PER_CLASS  = 10   # 10 x 11 = 110 images

if not ZERO_SHOT_CHECK:
    print("Zero-shot baseline skipped (ZERO_SHOT_CHECK=False).")
else:
    import random as _r2, collections as _col2
    _r2.seed(SEED + 99)

    print("Loading base model (NO adapter) for zero-shot comparison...")
    base_only = AutoModelForImageTextToText.from_pretrained(
        MODEL_NAME, torch_dtype=compute_dtype,
        device_map="auto" if DEVICE == "cuda" else None,
    )
    if DEVICE != "cuda": base_only = base_only.to(DEVICE)
    base_only.eval()
    print("Base model ready (zero-shot).")

    def classify_image_base(image):
        image = image.convert("RGB")
        pt = processor.apply_chat_template(build_prompt_messages(), tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[pt], images=[image], padding=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
        inputs = {k: v.to(next(base_only.parameters()).device) if hasattr(v, "to") else v for k, v in inputs.items()}
        with torch.inference_mode():
            gen_ids = base_only.generate(**inputs, max_new_tokens=64, do_sample=False)
        gen_ids = gen_ids[:, inputs["input_ids"].shape[1]:]
        return processor.batch_decode(gen_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

    test_split_full = dataset[EVAL_SPLIT]
    if MAX_SAMPLES:
        test_split_full = test_split_full.select(range(min(MAX_SAMPLES, len(test_split_full))))

    class_to_test_idx = _col2.defaultdict(list)
    for i, ex in enumerate(test_split_full):
        class_to_test_idx[id2label[int(ex["label"])]].append(i)

    sampled_zs_idx = []
    for label in new_labels:
        pool = class_to_test_idx[label]
        sampled_zs_idx.extend(_r2.sample(pool, min(N_ZS_PER_CLASS, len(pool))))

    zs_true, zs_pred_base, lora_pred_same = [], [], []

    for idx in tqdm(sampled_zs_idx, desc="Zero-shot inference"):
        ex  = test_split_full[idx]
        raw = classify_image_base(ex["image"])
        zs_true.append(id2label[int(ex["label"])])
        zs_pred_base.append(normalize_prediction(raw))
        lora_pred_same.append(pred_labels_list[idx])

    zs_t1   = sum(t == p for t, p in zip(zs_true, zs_pred_base))   / len(zs_true)
    lora_t1 = sum(t == p for t, p in zip(zs_true, lora_pred_same)) / len(zs_true)
    zs_fmt  = sum(has_valid_answer(r) for r in raw_responses[:len(zs_true)]) / len(zs_true)

    print("\n==== Zero-Shot vs LoRA (same stratified test sample) ====")
    print(f"  Sample size: {len(zs_true)} images  ({N_ZS_PER_CLASS} per class)")
    print(f"\n  {'Metric':14s} {'0.8B Zero-Shot':>16s} {'0.8B + LoRA':>14s} {'Gain':>10s}")
    print("  " + "─" * 58)
    print(f"  {'Top-1':14s} {zs_t1:>16.4f} {lora_t1:>14.4f} {lora_t1-zs_t1:>+10.4f}")

    zs_pc = {}; lora_pc_s = {}
    for label in new_labels:
        idxs = [i for i, t in enumerate(zs_true) if t == label]
        if idxs:
            zs_pc[label]      = sum(1 for i in idxs if zs_pred_base[i]   == label) / len(idxs)
            lora_pc_s[label]  = sum(1 for i in idxs if lora_pred_same[i] == label) / len(idxs)

    print(f"\n  {'Class':30s} {'ZeroShot':>9s} {'LoRA':>8s} {'Gain':>8s}")
    print("  " + "─" * 58)
    for label in new_labels:
        zv = zs_pc.get(label, float("nan"))
        lv = lora_pc_s.get(label, float("nan"))
        g  = lv - zv if not (np.isnan(zv) or np.isnan(lv)) else float("nan")
        print(f"  {label:30s} {zv:>9.3f} {lv:>8.3f} {g:>+8.3f}")

    labels_both = [l for l in new_labels if l in zs_pc]
    y = np.arange(len(labels_both)); bh = 0.35
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.barh(y - bh/2, [zs_pc[l]     for l in labels_both], bh, label="0.8B Zero-Shot",        color="#E74C3C", alpha=0.85, edgecolor="black", lw=0.5)
    ax.barh(y + bh/2, [lora_pc_s[l] for l in labels_both], bh, label="0.8B + LoRA (adapter)", color="#27AE60", alpha=0.85, edgecolor="black", lw=0.5)
    ax.set_yticks(y); ax.set_yticklabels(labels_both)
    ax.set_xlabel("Top-1 Accuracy"); ax.set_xlim(0, 1.15)
    ax.set_title(f"Zero-Shot vs LoRA — Per-Class Top-1\nZS={zs_t1:.3f}  |  LoRA={lora_t1:.3f}  |  Gain={lora_t1-zs_t1:+.3f}")
    ax.legend(); ax.invert_yaxis(); plt.tight_layout()
    zs_path = os.path.join(OUTPUT_DIR, f"zeroshot_vs_lora_{TAG}.png")
    plt.savefig(zs_path, dpi=150); plt.show()
    print(f"Saved: {zs_path}")

    del base_only
    if DEVICE == "cuda": torch.cuda.empty_cache()
    gc.collect()
    print("Base model unloaded.")


## 18. Per-Source-Dataset Breakdown (if path metadata available)

In [ ]:
if "predictions_df" not in dir():
    print("Run Section 10 first to build predictions_df.")
else:
    source_col     = predictions_df["source_dataset"]
    unique_sources = source_col.unique()

    if set(unique_sources) == {"unknown"}:
        print("Source dataset metadata not recovered. Showing class distribution:")
        print(pd.crosstab(predictions_df["true_label"], predictions_df["correct"],
                          margins=True).rename(columns={False: "Wrong", True: "Correct"}).to_string())
    else:
        print("Per-source-dataset performance:")
        rows = []
        for src in sorted(unique_sources):
            sub = predictions_df[source_col == src]
            rows.append({"source": src, "n": len(sub), "top1": sub["correct"].mean()})

        src_df = pd.DataFrame(rows).set_index("source")
        print(src_df.to_string(float_format="{:.4f}".format))

        fig, ax = plt.subplots(figsize=(8, 4))
        src_df["top1"].plot.bar(ax=ax, color="#3498DB", edgecolor="black")
        ax.set_title("Top-1 Accuracy by Source Dataset")
        ax.set_ylabel("Top-1"); ax.set_ylim(0, 1.05)
        ax.axhline(accuracy_top1, color="red", linestyle="--", label="Overall")
        ax.legend(); plt.tight_layout()
        src_path = os.path.join(OUTPUT_DIR, f"per_source_accuracy_{TAG}.png")
        plt.savefig(src_path, dpi=150); plt.show()
        print(f"Saved: {src_path}")
